In [ ]:
import pandas as pd
import json
import re
#from pydantic import BaseModel
from typing import Dict, List, Optional

In [ ]:
# --- Run configuration ---
from pathlib import Path
import os
import pandas as pd

# Set this to the run folder produced by your inference/eval notebooks
RUN_ID = os.getenv("RUN_ID", "20260322T165311Z_base_run")  # e.g., "2026-02-13_seed_ft_v3"
RUN_DIR = Path("../runs") / RUN_ID

# Input produced by 05_Recipie_Model_evaluation.ipynb (recommended)
# Fallback: you can point to predictions.parquet as well.
INPUT_FILE = RUN_DIR / "eval_scored.parquet"   # preferred
if not INPUT_FILE.exists():
    alt = RUN_DIR / "predictions.parquet"
    if alt.exists():
        INPUT_FILE = alt
    else:
        raise FileNotFoundError(f"Could not find eval_scored.parquet or predictions.parquet in {RUN_DIR}")

OUTPUT_FILE = RUN_DIR / "judge_results.parquet"     # intermediate/resumable
FINAL_FILE  = RUN_DIR / "final_eval.parquet"        # merged outputs + all metrics

RUN_DIR.mkdir(parents=True, exist_ok=True)

print("RUN_DIR:", RUN_DIR.resolve())
print("INPUT_FILE:", INPUT_FILE.resolve())

# --- Load input table (parquet or csv) ---
if INPUT_FILE.suffix.lower() == ".parquet":
    df = pd.read_parquet(INPUT_FILE)
else:
    df = pd.read_csv(INPUT_FILE)

df.head()



In [ ]:
# --- Standardize columns expected by the judge ---
# We will create these columns if they do not exist:
#   title (str), ingredients (list[str]), reference (list[str] OR str), prediction (str), recipe_id (str/int)

import numpy as np
import re

def _first_existing_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

# Resolve key fields from whatever is present
col_recipe_id = _first_existing_col(df, ["recipe_id", "id", "recipeID"])
col_title     = _first_existing_col(df, ["title_normalized", "title", "recipe_title"])
col_pred      = _first_existing_col(df, ["output", "model_output", "prediction", "generated", "instructions"])
col_ref       = _first_existing_col(df, ["directions_normalized", "reference_directions", "reference", "ground_truth"])
col_ing_list  = _first_existing_col(df, ["ingredients_ner", "ingredients_names_only", "ingredients_normalized", "ingredients_bullets"])

if col_pred is None:
    raise ValueError("Could not find a prediction column. Expected one of: output/model_output/prediction/generated/instructions")
if col_ref is None:
    raise ValueError("Could not find a reference column. Expected one of: directions_normalized/reference_directions/reference/ground_truth")
if col_ing_list is None:
    raise ValueError("Could not find an ingredients column. Expected one of: ingredients_names_only/ingredients_normalized/ingredients_bullets/ingredients")

# Create canonical columns
if col_recipe_id is None:
    df["recipe_id"] = np.arange(len(df))
else:
    df["recipe_id"] = df[col_recipe_id]

df["title"] = df[col_title].fillna("") if col_title else ""

df["prediction"] = df[col_pred].fillna("").astype(str)

# Reference can be list-of-steps or a string; normalize to list[str] for prompting
def _to_steps(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return []
    if isinstance(x, list):
        return [str(s).strip() for s in x if str(s).strip()]
    s = str(x).strip()
    if not s:
        return []
    # split numbered steps or sentences
    parts = re.split(r"\n+|(?:(?:^|\n)\s*\d+\.|\bStep\s*\d+\b)", s)
    parts = [p.strip(" .\t") for p in parts if p.strip()]
    return parts if len(parts) > 1 else [s]

df["reference"] = df[col_ref].apply(_to_steps)

# Ingredients: normalize to list[str]
def _to_ingredients(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return []
    if isinstance(x, list):
        return [str(i).strip("- ").strip() for i in x if str(i).strip()]
    s = str(x)
    # If bullet text, split lines
    lines = [ln.strip() for ln in s.splitlines() if ln.strip()]
    # Remove bullet prefixes
    lines = [re.sub(r"^[-*•]\s*", "", ln).strip() for ln in lines]
    return [ln for ln in lines if ln]

df["ingredients"] = df[col_ing_list].apply(_to_ingredients)

# If your input table still contains chat-formatted "input" and you prefer to parse from it,
# keep the parsing code below as a fallback. Otherwise, this standardized mapping is enough.

df[["recipe_id","title","ingredients","reference","prediction"]].head()


In [ ]:
df[["title", "ingredients"]].head()


In [ ]:
df.iloc[0]["ingredients"]

In [ ]:
df.iloc[0]['prediction']

In [ ]:
SYSTEM_MESSAGE = """
You are a professional chef, food writer, and culinary evaluator.
Your task is to objectively evaluate a generated recipe by comparing it against a reference recipe (ground truth).
You must be fair, consistent, and avoid personal preference.

As an expert culinary evaluator, assess the recipe using these criteria:
1. Ingredient Completeness - Are ingredients sufficient and appropriate compared to reference?
2. Instruction Clarity - Are steps clear, ordered, and actionable compared to reference?
3. Cooking Logic - Do steps follow correct culinary techniques compared to reference?
4. Recipe Coherence - Do ingredients and instructions align with reference?
5. Practicality - Can a home cook execute this recipe (vs reference)?
6. Originality - Does it avoid generic phrasing while staying true to reference?
7. Safety & Accuracy - No unsafe practices compared to reference standards
8. Reference Alignment - How closely does it match the reference recipe?

Return ONLY a JSON response with this format:
{
  "ingredient_completeness": number (1-5),
  "instruction_clarity": number (1-5),
  "cooking_logic": number (1-5),
  "recipe_coherence": number (1-5),
  "practicality": number (1-5),
  "originality": number (1-5),
  "safety_accuracy": number (1-5),
  "reference_alignment": number (1-5),
  "overall_score": number (1-5),
  "verdict": "PASS or FAIL",
  "short_feedback": "1-2 sentence summary",
  "key_differences": ["list of main differences from reference"]
}

GUIDELINES:
- Score 1 (poor) to 5 (excellent) for each criterion
- Overall score should reflect the average of all criteria
- Verdict is PASS only if all criteria score ≥3 and no critical failures
- Compare generated recipe against reference recipe for accuracy
- Note missing ingredients, extra ingredients, or substitutions
- Check if instructions follow the same logical flow as reference
- Identify any deviations that affect the recipe outcome
- Keep feedback concise and constructive
- Base evaluation on culinary standards, not personal taste
- Consider both professional and home cooking contexts
- Flag any potential food safety concerns
- Ensure measurements and times are reasonable compared to reference
"""

PROMPT_TEMPLATE = """
### TASK DATA
RECIPE TITLE:
{recipe_title}

INPUT INGREDIENTS PROVIDED TO MODEL:
{ingredients}

### REFERENCE (GROUND TRUTH)
REFERENCE RECIPE (Ground Truth):
{reference}

### GENERATED OUTPUT (TO BE EVALUATED)
GENERATED RECIPE TO EVALUATE:
{instructions}

Compare the generated recipe against the reference recipe and evaluate using the specified criteria. Return ONLY the JSON response with no additional text or explanation.
""".strip()

Configure LLM call with openrouter API key and model

In [ ]:
from openai import OpenAI
import os
from dotenv import load_dotenv
from pathlib import Path

# Load environment variables from .env file
env_path = Path.cwd() / ".env"
load_dotenv(env_path)

# Initialize OpenRouter client
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
  )
  
# Choose your judge model from OpenRouter
judge_model = "qwen/qwen-2.5-7b-instruct"#"anthropic/claude-3.5-sonnet"  # or "openai/gpt-4o", "meta-llama/llama-3.1-70b-instruct", etc.

Check a few samples from the evaluation dataframe to see how the metrics look.

In [ ]:
from pydantic import BaseModel

class EvaluationResult(BaseModel):
    reason: str
    is_correct: bool
    
def evaluate_recipe(
        recipe_title: str,
        ingredients: list[str],
        reference: list[str],  # ← Added reference
        instructions: str,
        *,
        strip_reasoning: bool = True
    ) -> tuple[bool, str, dict]:
    """
    Evaluates a generated recipe against a reference recipe.
    """
    
    # Format reference as a string
    reference_str = "\n".join(f"- {step}" for step in reference)
    
    # Format ingredients as a string
    ingredients_str = "\n".join(f"- {ing}" for ing in ingredients)
    judge_prompt = PROMPT_TEMPLATE.format(
        recipe_title=recipe_title,
        reference=reference_str,
        ingredients=ingredients_str,
        instructions=instructions
    )

    # Call OpenRouter
    completion = client.chat.completions.create(
        model=judge_model,
        messages=[
            {"role": "system", "content": SYSTEM_MESSAGE},
            {"role": "user",   "content": judge_prompt}
        ],
        temperature=0.0,
        top_p=1.0,
        max_tokens=8192,
        extra_headers={
            "HTTP-Referer": "http://localhost",
            "X-Title": "Recipe Evaluator"
        }
    )

    judge_reply = completion.choices[0].message.content

    # Parse JSON response
    try:
        evaluation = json.loads(judge_reply)
        is_pass = evaluation.get("verdict") == "PASS"
        reason = evaluation.get("short_feedback", "")
        return is_pass, reason, evaluation,completion
    except json.JSONDecodeError:
        return False, f"Could not parse JSON response.\nRaw reply:\n{judge_reply}", {}

In [ ]:
is_pass, reason, evaluation,completion = evaluate_recipe(
    recipe_title=df.iloc[10]["title"],
    ingredients=df.iloc[10]["ingredients"],
    reference= df.iloc[10]['reference'],
    instructions= df.iloc[10]['prediction']

)

print(f"Verdict: {'PASS' if is_pass else 'FAIL'}")
print(f"Feedback: {reason}")
print(f"Scores: {evaluation}")

In [ ]:
df.iloc[10]["title"]

In [ ]:
df.iloc[10]['reference']

In [ ]:
df.iloc[10]['prediction']

Configure for Asynchronous API calls.( via openrouter)

In [ ]:
import asyncio
import json
import pandas as pd
from openai import AsyncOpenAI
from pathlib import Path
import re
import numpy as np
import os
import time

async_client = AsyncOpenAI(
    base_url='https://openrouter.ai/api/v1',
    api_key=os.getenv('OPENROUTER_API_KEY')
)

# --- controls ---
MAX_ROWS = int(os.getenv('JUDGE_MAX_ROWS', '10000'))      # evaluate first N rows (for iteration); set large for full run
BATCH_SIZE = int(os.getenv('JUDGE_BATCH_SIZE', '25'))    # keep conservative to avoid rate limits
RESUME = os.getenv('JUDGE_RESUME', '1') == '1'
PARQUET_RETRY_COUNT = int(os.getenv('JUDGE_PARQUET_RETRIES', '3'))

RESULT_COLUMNS = [
    'recipe_id',
    'ingredient_completeness',
    'instruction_clarity',
    'cooking_logic',
    'recipe_coherence',
    'practicality',
    'originality',
    'safety_accuracy',
    'reference_alignment',
    'overall_score',
    'verdict',
    'short_feedback',
    'key_differences',
    '_raw_response',
]

def _extract_json(text: str) -> dict:
    if text is None:
        raise ValueError('Empty judge response')
    t = text.strip()
    t = re.sub(r'^```(?:json)?\s*|```\s*$', '', t, flags=re.IGNORECASE | re.MULTILINE).strip()
    try:
        return json.loads(t)
    except Exception:
        pass
    m = re.search(r'\{.*\}', t, flags=re.DOTALL)
    if not m:
        raise ValueError('No JSON object found in judge response')
    return json.loads(m.group(0))

def _normalize_key_differences(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return []
    if isinstance(x, list):
        return [str(v).strip() for v in x if str(v).strip()]
    if isinstance(x, tuple):
        return [str(v).strip() for v in x if str(v).strip()]
    if isinstance(x, str):
        s = x.strip()
        if not s:
            return []
        if s.startswith('[') and s.endswith(']'):
            try:
                parsed = json.loads(s)
                if isinstance(parsed, (list, tuple)):
                    return [str(v).strip() for v in parsed if str(v).strip()]
            except Exception:
                pass
        return [part.strip() for part in s.split(';') if part.strip()]
    return [str(x).strip()] if str(x).strip() else []

def normalize_judge_output(row: dict) -> dict:
    out = {c: pd.NA for c in RESULT_COLUMNS}
    if isinstance(row, dict):
        for c in RESULT_COLUMNS:
            if c in row:
                out[c] = row[c]
    out['key_differences'] = json.dumps(_normalize_key_differences(out.get('key_differences', [])), ensure_ascii=False)
    return out


def _save_with_retries(df: pd.DataFrame, path: Path, retries: int = PARQUET_RETRY_COUNT):
    delay = 1.0
    for attempt in range(1, retries + 1):
        try:
            df.to_parquet(path, index=False)
            return True
        except Exception as e:
            if attempt >= retries:
                raise
            print(f'Parquet write failed (attempt {attempt}/{retries}): {e}')
            time.sleep(delay)
            delay *= 2
    return False

async def evaluate_recipe_async(row):
    ingredients_str = '\n'.join(f'- {ing}' for ing in row['ingredients'])
    reference_str   = '\n'.join(f'- {step}' for step in row['reference'])

    judge_prompt = PROMPT_TEMPLATE.format(
        recipe_title=row.get('title', ''),
        reference=reference_str,
        ingredients=ingredients_str,
        instructions=row.get('prediction', '')
    )

    completion = await async_client.chat.completions.create(
        model=judge_model,
        messages=[
            {'role': 'system', 'content': SYSTEM_MESSAGE},
            {'role': 'user', 'content': judge_prompt}
        ],
        temperature=0.0,
        max_tokens=2048,
        extra_headers={
            'HTTP-Referer': 'http://localhost',
            'X-Title': 'Recipe Evaluator'
        }
    )

    raw = completion.choices[0].message.content
    data = _extract_json(raw)
    data['_raw_response'] = raw
    data['recipe_id'] = row['recipe_id']
    return normalize_judge_output(data)

async def process_batch(df_batch: pd.DataFrame):
    tasks = [evaluate_recipe_async(r) for r in df_batch.to_dict(orient='records')]
    return await asyncio.gather(*tasks)

# --- Resume support ---
already = set()
if RESUME and OUTPUT_FILE.exists():
    prev = pd.read_parquet(OUTPUT_FILE)
    if 'recipe_id' in prev.columns:
        already = set(prev['recipe_id'].tolist())
    print(f'Resuming: found {len(already)} already-judged rows in {OUTPUT_FILE.name}')

# --- Select rows to judge ---
df_run = df.head(min(MAX_ROWS, len(df))).copy()
df_run = df_run[~df_run['recipe_id'].isin(already)].copy()
print('Rows to judge this run:', len(df_run))

all_rows = []
for i in range(0, len(df_run), BATCH_SIZE):
    batch = df_run.iloc[i:i+BATCH_SIZE]
    try:
        batch_results = await process_batch(batch)
    except Exception as e:
        print(f'Batch failed at rows {i}..{i+len(batch)-1}: {e}')
        # continue after failure (optional): skip batch
        continue

    all_rows.extend(batch_results)

    # incremental save for safety
    out_df = pd.DataFrame(all_rows)
    if RESUME and OUTPUT_FILE.exists():
        prev = pd.read_parquet(OUTPUT_FILE).reindex(columns=RESULT_COLUMNS, fill_value=pd.NA)
        out_df = pd.concat([prev, out_df], ignore_index=True).drop_duplicates(subset=['recipe_id'], keep='last')
    out_df = out_df[RESULT_COLUMNS]
    _save_with_retries(out_df, OUTPUT_FILE)
    print(f'Completed batch {(i//BATCH_SIZE)+1} | saved {len(out_df)} rows to {OUTPUT_FILE.name}')

print('Done. judge_results rows:', pd.read_parquet(OUTPUT_FILE).shape[0])


In [ ]:
del(df_eval_results)

In [ ]:
# --- Canonical post-judge analysis setup ---
import warnings
from collections import Counter
import ast
import json
import pandas as pd
import plotly.graph_objects as go

categories = [
    "ingredient_completeness",
    "instruction_clarity",
    "cooking_logic",
    "recipe_coherence",
    "practicality",
    "originality",
    "safety_accuracy",
    "reference_alignment",
]

required_cols = ["recipe_id", "verdict"] + categories + ["overall_score"]


def normalize_judge_results(df_eval_results: pd.DataFrame) -> pd.DataFrame:
    """Normalize judge results for deterministic analysis."""
    out = df_eval_results.copy()

    # Ensure required columns exist.
    for c in required_cols:
        if c not in out.columns:
            out[c] = pd.NA

    # Normalize score columns to numeric.
    for c in categories + ["overall_score"]:
        out[c] = pd.to_numeric(out[c], errors="coerce")

    # Normalize verdict values.
    out["verdict"] = out["verdict"].astype(str).str.strip().str.upper()

    # Normalize key_differences to list[str].
    def _to_list(x):
        if x is None or (isinstance(x, float) and pd.isna(x)):
            return []
        if isinstance(x, list):
            return [str(v).strip() for v in x if str(v).strip()]
        if isinstance(x, tuple):
            return [str(v).strip() for v in x if str(v).strip()]
        if isinstance(x, str):
            s = x.strip()
            if not s:
                return []
            # Try parsing serialized list first.
            if s.startswith("[") and s.endswith("]"):
                try:
                    parsed = ast.literal_eval(s)
                    if isinstance(parsed, (list, tuple)):
                        return [str(v).strip() for v in parsed if str(v).strip()]
                except Exception:
                    pass
            # Fallback: split by semicolon.
            return [part.strip() for part in s.split(";") if part.strip()]
        return [str(x).strip()] if str(x).strip() else []

    if "key_differences" not in out.columns:
        out["key_differences"] = [[] for _ in range(len(out))]
    else:
        out["key_differences"] = out["key_differences"].apply(_to_list)

    if "short_feedback" not in out.columns:
        out["short_feedback"] = ""
    out["short_feedback"] = out["short_feedback"].fillna("").astype(str)

    out["key_differences_str"] = out["key_differences"].apply(lambda xs: "; ".join(xs))
    return out


def validate_analysis_inputs(df_eval_results: pd.DataFrame):
    missing = [c for c in ["recipe_id", "verdict", "overall_score"] if c not in df_eval_results.columns]
    if missing:
        raise ValueError(f"Missing required columns for analysis: {missing}")


def warn_on_sparse_scores(df_eval_results: pd.DataFrame):
    for c in categories + ["overall_score"]:
        null_ratio = float(df_eval_results[c].isna().mean())
        if null_ratio > 0.8:
            warnings.warn(f"Column {c} is mostly null ({null_ratio:.1%}).")


df_eval_results = pd.read_parquet(OUTPUT_FILE)
df_eval_results = normalize_judge_results(df_eval_results)
validate_analysis_inputs(df_eval_results)
warn_on_sparse_scores(df_eval_results)

# Canonical summary stats used by later cells.
averages = df_eval_results[categories + ["overall_score"]].mean(numeric_only=True)
pass_rate = df_eval_results["verdict"].value_counts(normalize=True, dropna=False) * 100
correlations = df_eval_results[["instruction_clarity", "cooking_logic", "safety_accuracy"]].corr()

print(f"Loaded judge results rows: {len(df_eval_results):,}")


In [ ]:
print("Average scores:")
print(averages)
print("\nPass/Fail rate (%):")
print(pass_rate)
print("\nSelected correlations:")
print(correlations)

df_eval_results.head()


In [ ]:
# Radar chart: average judge dimensions.
fig = go.Figure()
fig.add_trace(go.Scatterpolar(
    r=averages[categories].values,
    theta=categories,
    fill='toself',
    name='Model Average',
))
fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 5])),
    showlegend=True,
    title="Fine-tuned Model: Culinary Capability Profile",
)
fig.show()

# Top qualitative failures from normalized key_differences list.
all_diffs = [d for diffs in df_eval_results["key_differences"] for d in diffs]
common_errors = Counter(all_diffs).most_common(10)
print("Top 10 Culinary Failures:")
for error, count in common_errors:
    print(f"- {error}: {count} occurrences")


In [ ]:
# --- Merge judge results into the main table and save final artifact ---
df_merged = pd.merge(df, df_eval_results, on="recipe_id", how="left", suffixes=("", "_judge"))
df_merged.to_parquet(FINAL_FILE, index=False)

matched = int(df_merged["overall_score"].notna().sum())
coverage = (matched / max(len(df_merged), 1)) * 100.0
print(f"Saved merged final eval to {FINAL_FILE}")
print(f"Merge coverage: matched={matched:,} / total={len(df_merged):,} ({coverage:.1f}%)")

df_merged.head()


C. Distribution of scores (detect judge bias)


In [ ]:
df_merged["overall_score"].value_counts(dropna=False).sort_index()


2. PASS vs FAIL score comparison


In [ ]:
df_merged.groupby("verdict", dropna=False)[categories].mean(numeric_only=True)


4. Weakest-dimension finder


In [ ]:
score_view = df_merged[categories].copy()
df_merged["weakest_dimension"] = score_view.idxmin(axis=1)
df_merged["weakest_dimension"].value_counts(dropna=False)


Qualitative Analysis: Most common failure reasons


In [ ]:
pd.Series([d for diffs in df_merged["key_differences"].fillna('').apply(lambda x: x if isinstance(x, list) else [s.strip() for s in str(x).split(';') if s.strip()]) for d in diffs]).value_counts().head(10)


In [ ]:
# Keep merged table intact; avoid global dropna.
df_merged.info()
